# IMU Fall/Activity Detection - Baseline Exploration

This notebook provides an interactive exploration of the IMU sensor data and baseline ML models for fall/activity detection.

## Setup


In [ ]:
import sys
import os

# Add parent directory for imports
sys.path.insert(0, os.path.dirname(os.getcwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ml.imu_fall_detection_pipeline import (
    load_all_sessions,
    make_windows,
    extract_features_for_windows,
    train_test_split_by_session,
    train_logistic_regression,
    train_random_forest,
    evaluate_model
)

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')


## 1. Load Data


In [ ]:
# Path to collected data
DATA_DIR = "../backend/collected_data"

# Load all sessions
df = load_all_sessions(DATA_DIR)
df.head(10)


In [ ]:
# Data overview
print(f"Total samples: {len(df):,}")
print(f"Sessions: {df['session_id'].nunique()}")
print(f"Labels: {df['label'].nunique()}")
print(f"\nLabel distribution:")
df['label'].value_counts()


## 2. Explore Label Distribution


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Label counts
ax1 = axes[0]
label_counts = df['label'].value_counts()
label_counts.plot(kind='barh', ax=ax1, color=sns.color_palette('husl', len(label_counts)))
ax1.set_xlabel('Sample Count')
ax1.set_ylabel('Activity Label')
ax1.set_title('Sample Distribution by Activity')
ax1.invert_yaxis()

# Samples per session
ax2 = axes[1]
session_counts = df.groupby('session_id').size()
session_counts.plot(kind='bar', ax=ax2, color='steelblue')
ax2.set_xlabel('Session ID')
ax2.set_ylabel('Sample Count')
ax2.set_title('Samples per Session')
ax2.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()


## 3. Visualize Raw Sensor Signals


In [ ]:
def plot_session_signals(df, session_id, time_range=None):
    """Plot accelerometer and gyroscope signals for a session."""
    session_data = df[df['session_id'] == session_id].copy()
    
    # Normalize time to start from 0
    session_data['time_s'] = (session_data['timestamp_ms'] - session_data['timestamp_ms'].min()) / 1000
    
    if time_range:
        session_data = session_data[(session_data['time_s'] >= time_range[0]) & 
                                     (session_data['time_s'] <= time_range[1])]
    
    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
    
    # Compute magnitudes
    session_data['a_mag'] = np.sqrt(session_data['ax']**2 + session_data['ay']**2 + session_data['az']**2)
    session_data['w_mag'] = np.sqrt(session_data['gx']**2 + session_data['gy']**2 + session_data['gz']**2)
    
    # Plot accelerometer
    ax1 = axes[0]
    ax1.plot(session_data['time_s'], session_data['ax'], label='ax', alpha=0.7)
    ax1.plot(session_data['time_s'], session_data['ay'], label='ay', alpha=0.7)
    ax1.plot(session_data['time_s'], session_data['az'], label='az', alpha=0.7)
    ax1.plot(session_data['time_s'], session_data['a_mag'], label='|a|', linewidth=2, color='black')
    ax1.set_ylabel('Acceleration (m/s²)')
    ax1.legend(loc='upper right')
    ax1.set_title(f'Session {session_id} - Accelerometer')
    ax1.grid(True, alpha=0.3)
    
    # Plot gyroscope
    ax2 = axes[1]
    ax2.plot(session_data['time_s'], session_data['gx'], label='gx', alpha=0.7)
    ax2.plot(session_data['time_s'], session_data['gy'], label='gy', alpha=0.7)
    ax2.plot(session_data['time_s'], session_data['gz'], label='gz', alpha=0.7)
    ax2.plot(session_data['time_s'], session_data['w_mag'], label='|ω|', linewidth=2, color='black')
    ax2.set_ylabel('Angular velocity (rad/s)')
    ax2.legend(loc='upper right')
    ax2.set_title(f'Session {session_id} - Gyroscope')
    ax2.grid(True, alpha=0.3)
    
    # Plot labels
    ax3 = axes[2]
    labels_encoded = pd.Categorical(session_data['label']).codes
    ax3.scatter(session_data['time_s'], labels_encoded, c=labels_encoded, cmap='tab10', s=5, alpha=0.7)
    ax3.set_ylabel('Activity Label')
    ax3.set_xlabel('Time (seconds)')
    ax3.set_title(f'Session {session_id} - Activity Labels')
    
    # Add label names to y-axis
    unique_labels = session_data['label'].unique()
    ax3.set_yticks(range(len(unique_labels)))
    ax3.set_yticklabels(unique_labels)
    
    plt.tight_layout()
    plt.show()

# Plot first session
plot_session_signals(df, session_id=1)


## 4. Create Windows & Extract Features


In [ ]:
# Create windows and extract features
WINDOW_SIZE_S = 1.0
WINDOW_STEP_S = 0.5

windows_df = make_windows(df, window_size_s=WINDOW_SIZE_S, window_step_s=WINDOW_STEP_S)
features_df, X, y = extract_features_for_windows(df, windows_df)

print(f"Feature matrix shape: {X.shape}")
feature_cols = [c for c in features_df.columns if c not in ['session_id', 'window_idx', 'window_start_ms', 'window_end_ms', 'label', 'n_samples']]
print(f"Features: {feature_cols}")


## 5. Train/Test Split & Model Training


In [ ]:
# Split by session (prevents data leakage)
X_train, X_test, y_train, y_test, sessions_train, sessions_test = train_test_split_by_session(
    features_df, X, y, test_size=0.3, random_state=42
)

# Train models
print("\nTraining Logistic Regression...")
lr_model, lr_scaler = train_logistic_regression(X_train, y_train)

print("Training Random Forest...")
rf_model = train_random_forest(X_train, y_train)


## 6. Evaluation & Confusion Matrices


In [ ]:
# Evaluate both models
lr_results = evaluate_model(lr_model, X_test, y_test, "Logistic Regression", scaler=lr_scaler)
rf_results = evaluate_model(rf_model, X_test, y_test, "Random Forest")

# Plot confusion matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, results in zip(axes, [lr_results, rf_results]):
    sns.heatmap(results['confusion_matrix'], annot=True, fmt='d', cmap='Blues',
                xticklabels=results['labels'], yticklabels=results['labels'], ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(f"{results['model_name']}\nAccuracy: {results['accuracy']:.4f}")
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()


## 7. Feature Importance & Summary


In [ ]:
# Feature importance from Random Forest
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(range(len(feature_cols)), importances[indices], align='center')
ax.set_yticks(range(len(feature_cols)))
ax.set_yticklabels([feature_cols[i] for i in indices])
ax.invert_yaxis()
ax.set_xlabel('Feature Importance')
ax.set_title('Random Forest Feature Importances')
plt.tight_layout()
plt.show()

# Summary
print("\n" + "="*50)
print("MODEL COMPARISON SUMMARY")
print("="*50)
summary = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest'],
    'Accuracy': [f"{lr_results['accuracy']:.4f}", f"{rf_results['accuracy']:.4f}"],
    'Macro F1': [f"{lr_results['classification_report']['macro avg']['f1-score']:.4f}",
                 f"{rf_results['classification_report']['macro avg']['f1-score']:.4f}"]
})
print(summary.to_string(index=False))
